In [1]:
from xgboost import XGBClassifier
from pathlib import Path
import pandas as pd
from sklearn.model_selection import GridSearchCV

PROJECT_ROOT = Path.cwd()
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR = PROJECT_ROOT / "data" / "predictions"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

X_train_path = PROCESSED_DATA_DIR / "X_train.csv"
y_train_path = PROCESSED_DATA_DIR / "y_train.csv"
X_test_path  = PROCESSED_DATA_DIR / "X_test.csv"

X_train = pd.read_csv(X_train_path)
y_train = pd.read_csv(y_train_path)
X_test  = pd.read_csv(X_test_path)

train_ids = X_train['PassengerId']
test_ids  = X_test['PassengerId']

X_train = X_train.drop(columns=['PassengerId'])
X_test  = X_test.drop(columns=['PassengerId'])

param_grid = {
    'max_depth': [3, 5, 7],
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.1, 0.2],
    'reg_alpha': [0, 0.1],
    'reg_lambda': [1, 1.5],
    'gamma': [0, 0.1],
}

grid = GridSearchCV(
    XGBClassifier(random_state=42, eval_metric="error"),
    param_grid,
    scoring="accuracy",
    cv=5
)

grid.fit(X_train, y_train)

y_pred = grid.predict(X_test).astype(bool)

submission = pd.DataFrame({
    "PassengerId": test_ids,
    "Transported": y_pred
})

submission.to_csv(OUTPUT_DIR / "xgb_predictions.csv", index=False)


print("Predictions saved to:", OUTPUT_DIR / "xgb_predictions.csv")
print("=" * 50)
print("BEST MODEL INFORMATION")
print("=" * 50)

print(f"\nBest parameters found:")
for param, value in grid.best_params_.items():
    print(f"  {param}: {value}")

print(f"\nBest cross-validation score: {grid.best_score_:.4f}")
print(f"Best estimator index: {grid.best_index_}")

Predictions saved to: c:\Users\danci\Desktop\studia\MOA\titanic-moa\data\predictions\xgb_predictions.csv
BEST MODEL INFORMATION

Best parameters found:
  gamma: 0
  learning_rate: 0.1
  max_depth: 3
  n_estimators: 100
  reg_alpha: 0
  reg_lambda: 1.5

Best cross-validation score: 0.7935
Best estimator index: 37
